# Практическое занятие: Сегментация предложений (Sentence Boundary Disambiguation)
## Кейс: Анализ сложной авторской пунктуации Л.Н. Толстого («Война и мир»)

В этом ноутбуке мы сравним три поколения алгоритмов сегментации текста:
1. **Наивный подход на регулярных выражениях (RegEx)**
2. **Продвинутый подход на правилах (Rule-Based инструмент `pysbd`)**
3. **Статистический подход без учителя (Алгоритм Punkt из библиотеки `NLTK`)**
4. **Нейросетевой синтаксический подход (Библиотека `spaCy` на базе Deep Learning)**

### Цель:
Увидеть, как алгоритмы обрабатывают инверсию авторских слов («Что это? я падаю?...» — подумал он) и эмоциональные многоточия.

In [1]:
%pip install pysbd nltk spacy

!python -m spacy download ru_core_news_sm

  Using cached pysbd-0.3.4-py3-none-any.whl.metadata (6.1 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.15-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp314-cp314-macosx_11_0_arm64.whl.metadata (9.7 kB)
  Using cached preshed-3.0.13-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached thinc-8.3.13-cp314-cp314-macosx_11_0_arm64.whl.metadata (14 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (19 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-1.0.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached confection-1.3.3-py3-none-any.whl.metadata (19 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached blis-1.3.3-cp314-cp314-macosx_11_0_arm64.whl.met

In [2]:
# Целевой текст для анализа
text = (
    "«Что это? я падаю? у меня ноги подкашиваются», — подумал он и упал на спину. "
    "Он раскрыл глаза, надеясь увидать, чем кончилась борьба французов с артиллеристами, "
    "и желая знать, убит или нет рыжий артиллерист, взяты или спасены пушки. Но он ничего не видал. "
    "Над ним не было ничего уже, кроме неба,— высокого неба, не ясного, но все-таки неизмеримо высокого, "
    "с тихо ползущими по нем серыми облаками. «Как тихо, спокойно и торжественно, совсем не так, как я бежал,— "
    "подумал князь Андрей,— не так, как мы бежали, кричали и дрались; совсем не так, как с озлобленными и "
    "испуганными лицами тащили друг у друга банник француз и артиллерист,— совсем не так ползут облака по этому "
    "высокому бесконечному небу. Как же я не видал прежде этого высокого неба? И как я счастлив, что узнал его наконец. "
    "Да! все пустое, все обман, кроме этого бесконечного неба. Ничего, ничего нет, кроме его. Но и того даже нет, "
    "ничего нет, кроме тишины, успокоения. И слава богу!..»"
)

print(f"Длина исходного текста: {len(text)} символов.")

Длина исходного текста: 948 символов.


## Подход 1: Наивный RegEx (Разделение по знакам . ! ? + пробел)
Этот подход ищет любые маркеры конца предложения и сразу режет текст, не задумываясь о контексте.

In [3]:
import re

print("=== 1. РЕЗУЛЬТАТ НАИВНОГО REGEX ===")

# Паттерн: делим там, где после . ! или ? идет один или более пробелов
regex_sentences = re.split(r'(?<=[.!?])\s+', text)

for i, sent in enumerate(regex_sentences, 1):
    print(f"Предложение {i}: {sent}")


=== 1. РЕЗУЛЬТАТ НАИВНОГО REGEX ===
Предложение 1: «Что это?
Предложение 2: я падаю?
Предложение 3: у меня ноги подкашиваются», — подумал он и упал на спину.
Предложение 4: Он раскрыл глаза, надеясь увидать, чем кончилась борьба французов с артиллеристами, и желая знать, убит или нет рыжий артиллерист, взяты или спасены пушки.
Предложение 5: Но он ничего не видал.
Предложение 6: Над ним не было ничего уже, кроме неба,— высокого неба, не ясного, но все-таки неизмеримо высокого, с тихо ползущими по нем серыми облаками.
Предложение 7: «Как тихо, спокойно и торжественно, совсем не так, как я бежал,— подумал князь Андрей,— не так, как мы бежали, кричали и дрались; совсем не так, как с озлобленными и испуганными лицами тащили друг у друга банник француз и артиллерист,— совсем не так ползут облака по этому высокому бесконечному небу.
Предложение 8: Как же я не видал прежде этого высокого неба?
Предложение 9: И как я счастлив, что узнал его наконец.
Предложение 10: Да!
Предложение 11: все пуст

## Подход 2: Эвристический алгоритм на правилах (pysbd)
Библиотека **pysbd** использует каскад регулярных выражений и списки исключений. Она спроектирована так, чтобы изолировать знаки препинания внутри кавычек, обрабатывать сокращения и учитывать специфические правила конкретного языка. Обязательно инициализируем сегментатор с параметром `language="ru"`.


In [4]:
import pysbd

print("=== 2. РЕЗУЛЬТАТ ЭВРИСТИКИ PYSBD ===")

# Инициализируем сегментатор для русского языка (clean=False, чтобы не изменять оригинальный текст)
segmenter = pysbd.Segmenter(language="ru", clean=False)
pysbd_sentences = segmenter.segment(text)

for i, sent in enumerate(pysbd_sentences, 1):
    print(f"Предложение {i}: {sent}")


=== 2. РЕЗУЛЬТАТ ЭВРИСТИКИ PYSBD ===
Предложение 1: «Что это? я падаю? у меня ноги подкашиваются», — подумал он и упал на спину. 
Предложение 2: Он раскрыл глаза, надеясь увидать, чем кончилась борьба французов с артиллеристами, и желая знать, убит или нет рыжий артиллерист, взяты или спасены пушки. 
Предложение 3: Но он ничего не видал. 
Предложение 4: Над ним не было ничего уже, кроме неба,— высокого неба, не ясного, но все-таки неизмеримо высокого, с тихо ползущими по нем серыми облаками. 
Предложение 5: «Как тихо, спокойно и торжественно, совсем не так, как я бежал,— подумал князь Андрей,— не так, как мы бежали, кричали и дрались; совсем не так, как с озлобленными и испуганными лицами тащили друг у друга банник француз и артиллерист,— совсем не так ползут облака по этому высокому бесконечному небу. Как же я не видал прежде этого высокого неба? И как я счастлив, что узнал его наконец. Да! все пустое, все обман, кроме этого бесконечного неба. Ничего, ничего нет, кроме его. Но и того 

## Подход 3: Статистический алгоритм (NLTK Punkt)
Алгоритм без учителя, обучающийся «на лету». Он оценивает вероятность того, является ли знак препинания концом предложения, на основе накопленной статистики совместного появления слов и точек.

In [5]:
import nltk

# Загружаем необходимый ресурс для NLTK (алгоритм Punkt)
nltk.download('punkt_tab')

print("=== 3. РЕЗУЛЬТАТ СТАТИСТИЧЕСКОГО АЛГОРИТМА (NLTK) ===")

nltk_sentences = nltk.sent_tokenize(text, language="russian")

for i, sent in enumerate(nltk_sentences, 1):
    print(f"Предложение {i}: {sent}")


=== 3. РЕЗУЛЬТАТ СТАТИСТИЧЕСКОГО АЛГОРИТМА (NLTK) ===
Предложение 1: «Что это?
Предложение 2: я падаю?
Предложение 3: у меня ноги подкашиваются», — подумал он и упал на спину.
Предложение 4: Он раскрыл глаза, надеясь увидать, чем кончилась борьба французов с артиллеристами, и желая знать, убит или нет рыжий артиллерист, взяты или спасены пушки.
Предложение 5: Но он ничего не видал.
Предложение 6: Над ним не было ничего уже, кроме неба,— высокого неба, не ясного, но все-таки неизмеримо высокого, с тихо ползущими по нем серыми облаками.
Предложение 7: «Как тихо, спокойно и торжественно, совсем не так, как я бежал,— подумал князь Андрей,— не так, как мы бежали, кричали и дрались; совсем не так, как с озлобленными и испуганными лицами тащили друг у друга банник француз и артиллерист,— совсем не так ползут облака по этому высокому бесконечному небу.
Предложение 8: Как же я не видал прежде этого высокого неба?
Предложение 9: И как я счастлив, что узнал его наконец.
Предложение 10: Да!
Предло

[nltk_data] Downloading package punkt_tab to /Users/ksrve/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Подход 4: Нейросетевой подход (spaCy Dependency Parser)
Модель `spaCy` пропускает текст через сверточную нейросеть (CNN), которая одновременно предсказывает части речи (POS-tagging) и синтаксические связи (Dependency Tree). Граница предложения определяется на основе логической завершенности синтаксической структуры.

In [6]:
import spacy

# Загружаем легковесную нейросетевую модель spaCy для русского языка
nlp = spacy.load("ru_core_news_sm")

print("=== 4. РЕЗУЛЬТАТ НЕЙРОСЕТЕВОГО ПОДХОДА (SPACY) ===")

# Передаем текст в пайплайн spaCy. Нейросеть выполняет комплексный анализ
doc = nlp(text)

# Извлекаем предложения, которые распознала нейросеть
spacy_sentences = [sent.text.strip() for sent in doc.sents]

for i, sent in enumerate(spacy_sentences, 1):
    print(f"Предложение {i}: {sent}")

=== 4. РЕЗУЛЬТАТ НЕЙРОСЕТЕВОГО ПОДХОДА (SPACY) ===
Предложение 1: «Что это?
Предложение 2: я падаю?
Предложение 3: у меня ноги подкашиваются», — подумал он и упал на спину.
Предложение 4: Он раскрыл глаза, надеясь увидать, чем кончилась борьба французов с артиллеристами, и желая знать, убит или нет рыжий артиллерист, взяты или спасены пушки.
Предложение 5: Но он ничего не видал.
Предложение 6: Над ним не было ничего уже, кроме неба,— высокого неба, не ясного, но все-таки неизмеримо высокого, с тихо ползущими по нем серыми облаками.
Предложение 7: «Как тихо, спокойно и торжественно, совсем не так, как я бежал,— подумал князь Андрей,— не так, как мы бежали, кричали и дрались; совсем не так, как с озлобленными и испуганными лицами тащили друг у друга банник француз и артиллерист,— совсем не так ползут облака по этому высокому бесконечному небу.
Предложение 8: Как же я не видал прежде этого высокого неба?
Предложение 9: И как я счастлив, что узнал его наконец.
Предложение 10: Да!
Предложен